In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import sys

sys.path.append("..")

In [2]:
DATA_DIR = "../mu3e_trigger_data"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/run42_sig_pixel_128.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/run42_bg_pixel_128.npy"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_mppc_128.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_mppc_128.npy"

bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_seqlen = (bg_pixel_spacetime != -1).all(axis=-1).sum(axis=-1)
sig_pixel_seqlen = (sig_pixel_spacetime != -1).all(axis=-1).sum(axis=-1)

input_seq_len = bg_pixel_spacetime.shape[1]
input_dim = bg_pixel_spacetime.shape[2]  # Exclude timestamp

from src.utils import cartesian_to_cylindrical
bg_pixel_cylindrical = np.concatenate(
    [cartesian_to_cylindrical(bg_pixel_spacetime[:, :, :2]),
     bg_pixel_spacetime[:, :, 2:]], axis=-1)
sig_pixel_cylindrical = np.concatenate(
    [cartesian_to_cylindrical(sig_pixel_spacetime[:, :, :2]),
     sig_pixel_spacetime[:, :, 2:]], axis=-1)


In [3]:
from src.model.components import (
    MultiHeadAttentionBlock,
    MultiHeadAttentionStack,
    SelfAttentionBlock,
    SelfAttentionStack,
    PoolingAttentionBlock,
    GenerateMask,
    GetSequenceLength,
    DecoderQueries,
    GenerateDecoderMask,
    MLP,
    MaskOutput
)

feature_dim = 8
latent_dim = 128
dropout_rate = 0.0
num_seeds=latent_dim // feature_dim

pixel_input = keras.layers.Input(shape=(input_seq_len, input_dim), name="pixel_input")
pixel_mask = GenerateMask(name="pixel_mask")(pixel_input)
pixel_seqlen = GetSequenceLength(name="pixel_seqlen")(pixel_mask)

pixel_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_embedding",
)(pixel_input)

pixel_self_attention = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
)(pixel_embedding, pixel_mask)

pixel_pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=4,
    name="pixel_pooling",
)(pixel_self_attention, pixel_mask)

pixel_pooling_self_attention = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_pooling_self_attention",
)(pixel_pooling)

pixel_latent_space = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_latent_space",
)(pixel_pooling_self_attention)

pixel_seqlen_embedding = MLP(
    num_layers=2,
    output_dim=latent_dim,
    name="pixel_seqlen_embedding",
)(pixel_seqlen)

pixel_latent_output = keras.layers.Add(name="pixel_latent_output")(
    [
        keras.layers.Flatten(name="pixel_latent_flatten")(pixel_latent_space),
        pixel_seqlen_embedding,
    ]
)

decoder_seqlen = MLP(
    num_layers=2,
    output_dim=1,
    name="decoder_seqlen",
)(pixel_latent_output)


decoder_queries = DecoderQueries(
    num_queries=input_seq_len,
    feature_dim=feature_dim,
    name="decoder_queries",
)(decoder_seqlen)

decoded_embedded_queries = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="decoded_embedded_queries",
)(decoder_queries)

decoder_mask = GenerateDecoderMask(name="decoder_mask", max_length = 128)(decoder_seqlen)

decoded_latent_set = keras.layers.Reshape(
    (num_seeds, feature_dim), name="decoded_latent_space"
)(pixel_latent_output)

decoded_latent_set = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="decoded_latent_set",
)(decoded_latent_set)


decoded_latent_attention = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="decoded_latent_attention",
)(decoded_latent_set)


decoded_pixel_space = MultiHeadAttentionBlock(
    num_heads=4,
    key_dim=feature_dim,
    name="decoded_pixel_space",
)(decoded_embedded_queries, decoded_latent_attention, query_mask = decoder_mask)


decoded_point_set_self_attention = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="decoded_point_set_self_attention",
)(decoded_pixel_space, decoder_mask)

decoded_point_set = MLP(
    num_layers=4,
    output_dim=input_dim,
    name="decoded_point_set_mlp",
    activation="linear"
)(decoded_point_set_self_attention)

decoded_output = MaskOutput(
    name="decoded_output", padding_value = -1
)(decoded_point_set, decoder_mask)

seqlen_output = keras.layers.Concatenate(name="seqlen_output")(
    [
        keras.layers.Flatten(name="seqlen_flatten")(pixel_seqlen),
        keras.layers.Flatten(name="decoder_seqlen_flatten")(decoder_seqlen),
    ]
)

model = keras.Model(
    inputs=pixel_input,
    outputs=[decoded_output, seqlen_output],
    name="SetAutoEncoder",
)

from src.training import SplitMSE, ChamferDistanceMasked

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "decoded_output": ChamferDistanceMasked(mode="cylindrical"),
        "seqlen_output": SplitMSE(),
    },
)

model.summary()

Model: "SetAutoEncoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pixel_input         │ (None, 128, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_embedding     │ (None, 128, 8)    │        274 │ pixel_input[0][0] │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_mask          │ (None, 128, 1)    │          0 │ pixel_input[0][0] │
│ (GenerateMask)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_self_attenti… │ (None, 128, 8)    │      4,320 │ pixel_embedding[… │
│ (SelfAttentionStac… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_pooling       │ (None, 16, 8)     │      1,568 │ pixel_self_atten… │
│ (PoolingAttentionB… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_pooling_self… │ (None, 16, 8)     │      4,320 │ pixel_pooling[0]… │
│ (SelfAttentionStac… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_space  │ (None, 16, 8)     │        576 │ pixel_pooling_se… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_seqlen        │ (None, 1)         │          0 │ pixel_mask[0][0]  │
│ (GetSequenceLength) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_flatt… │ (None, 128)       │          0 │ pixel_latent_spa… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_seqlen_embed… │ (None, 128)       │      3,116 │ pixel_seqlen[0][… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_output │ (None, 128)       │          0 │ pixel_latent_fla… │
│ (Add)               │                   │            │ pixel_seqlen_emb… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_seqlen      │ (None, 1)         │      2,862 │ pixel_latent_out… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_latent_spa… │ (None, 16, 8)     │          0 │ pixel_latent_out… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_queries     │ (None, 128, 8)    │      1,024 │ decoder_seqlen[0… │
│ (DecoderQueries)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_latent_set  │ (None, 16, 8)     │        576 │ decoded_latent_s… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_embedded_q… │ (None, 128, 8)    │        576 │ decoder_queries[… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_latent_att… │ (None, 16, 8)     │      4,320 │ decoded_latent_s

 Total params: 25,715 (100.45 KB)

 Trainable params: 25,715 (100.45 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    x=bg_pixel_cylindrical,
    y={
        "decoded_output": bg_pixel_cylindrical,
        "seqlen_output": bg_pixel_seqlen,
    },
    batch_size=64,
    epochs=10,
    validation_split=0.1,
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
        )]
    ,
)

Epoch 1/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 192s 167ms/step - decoded_output_loss: 1944.6357 - loss: 2066.6274 - seqlen_output_loss: 121.9914 - val_decoded_output_loss: 885.4306 - val_loss: 885.2513 - val_seqlen_output_loss: 0.0019
Epoch 2/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 251s 230ms/step - decoded_output_loss: 875.5399 - loss: 875.5443 - seqlen_output_loss: 0.0046 - val_decoded_output_loss: 832.6817 - val_loss: 832.5546 - val_seqlen_output_loss: 0.0186
Epoch 3/10
 302/1094 ━━━━━━━━━━━━━━━━━━━━ 4:43 358ms/step - decoded_output_loss: 830.0145 - loss: 830.0292 - seqlen_output_loss: 0.0149

In [ ]:
pred = model.predict(x=sig_pixel_cylindrical[:1], batch_size=10)


In [ ]:
sig_pixel_cylindrical[:1]

In [ ]:
plt.scatter(
    sig_pixel_cylindrical[0, :, 0],
    sig_pixel_cylindrical[0, :, 1],
    c=sig_pixel_cylindrical[0, :, 2],
    cmap='viridis',
    s=10
)
pred = model.predict(x=sig_pixel_cylindrical[:1], batch_size=10)
plt.scatter(
    pred[0][0, :, 0],
    pred[0][0, :, 1],
    c=pred[0][0, :, 2],
    cmap='viridis',
    s=10
)